# 03b. Checkpoint and recovery lab

UTS 43008 · Assignment 3 · Project 12 · Dihedrals

Before the long study, train once without a break, train again with a pause, resume it and compare
the resulting state. This is a software test, not evidence of a useful strategy.

This follows the shared-helper structure in our Semester 3 Deep Learning Assignment 3. RL adds
game-state and random-state recovery because training experience is generated while the agent plays.

The default exercise performs only two 256-transition CPU learners. It uses no AWS services,
S3, final layouts or full training budgets.

In [ ]:
from pathlib import Path
import os
import sys
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path(os.environ.get("BASE_SIEGE_ROOT", Path.cwd())).expanduser()
ROOT = next((p for p in [candidate, *candidate.parents] if (p / "env/siege_env.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete project, or set BASE_SIEGE_ROOT to its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from env import config as C
from env.siege_env import SiegeEnv, describe_action, deployment_action
from env.render import draw_snapshot
from env.generator import generate_base
from experiments.provenance import runtime_manifest, sha256
RUNS = Path(os.environ.get("BASE_SIEGE_RUNS_DIR", ROOT / "runs")).expanduser().resolve()
OUTPUT = ROOT / "artifacts" / "notebook-sessions" / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
def save_figure(name):
    plt.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
print(f"Project: {ROOT.name} | Python: {sys.version.split()[0]}")
print(f"This notebook's outputs: {OUTPUT}")

## 1. Recovery point versus final model

| Artifact | Purpose |
|---|---|
| contract.json | Immutable settings, source, versions, input hashes and budgets |
| Checkpoint model.zip | Policy, value network and optimiser from a safe boundary |
| state.pt | Worker games, RNGs, last observations and development state |
| Checkpoint episodes.jsonl | History prefix consistent with those weights |
| checkpoint.json | Phase, completed steps and checksums |
| LATEST.json | Pointer to a published generation |
| Run-root model.zip | Completed fixed-budget model for inference |

These are Python serialised objects. Load only artifacts from trusted code and people.
Checksums detect damage; they do not make an unknown model safe.

In [ ]:
from tools.check_recovery import recovery_lab, compare_runs
from train.checkpoints import latest_checkpoint, verify_checkpoint
from train.train_ppo import resume

lab = OUTPUT / "recovery-lab"
report = recovery_lab(lab)
display(pd.DataFrame(report["checks"]).T)
assert report["passed"]

## 2. Inspect the evidence

The second learner pauses at 96 transitions during joint training, then resumes to 256.
The comparison checks every model and optimiser tensor, counters and canonical episode history.
Matching only the number of steps would not establish correct continuation.

In [ ]:
records = []
for label in ["continuous", "resumed"]:
    manifest = json.loads((lab / label / "manifest.json").read_text())
    records.append({"run": label, "status": manifest["status"], "actual_steps": manifest["actual_steps"],
        "attempts": len(manifest["attempts"]), "save_reload_verified": manifest["save_reload_verified"]})
display(pd.DataFrame(records))
display(pd.DataFrame(json.loads((lab / "resumed/manifest.json").read_text())["attempts"]))

## 3. Inspect a complete generation

The trainer writes a staging directory, flushes its files, renames it to a committed generation
and atomically updates the pointer. Incomplete staging directories are ignored.

If the newest generation is damaged, recovery can use an earlier intact one. That fallback may
repeat more work but preserves a consistent training state.

In [ ]:
checkpoint, metadata, rejected = latest_checkpoint(lab / "resumed")
assert verify_checkpoint(checkpoint) == metadata
display(pd.Series({k: metadata[k] for k in ["phase", "actual_steps", "optimizer_updates", "boundary"]}))
display(pd.DataFrame([{"file": name, "size_MiB": (checkpoint / name).stat().st_size / 1024**2,
    "sha256": digest} for name, digest in metadata["files"].items()]))
print("Rejected generations:", rejected)

## 4. Completed runs are not repeated

Resume again. It should return the completion record without adding another training attempt.
This matters when reopening and rerunning notebook cells.

In [ ]:
before = json.loads((lab / "resumed/manifest.json").read_text())
after = resume(lab / "resumed")
assert len(before["attempts"]) == len(after["attempts"])
print("No training repeated.")

## 5. Abrupt shutdown test

The exercise above pauses cleanly. This terminal command also kills its own test process after
a checkpoint is published, then verifies recovery. It never kills another run or the notebook kernel.

~~~bash
uv run python -m tools.check_recovery --out artifacts/studio-kill-check --kill-process
~~~

The test suite also covers a corrupt latest checkpoint, partial writes, two-worker PA, changed
source, moved run folders, and recovery of episode and development history.

In [ ]:
rows = []
for path in sorted((lab / "resumed/checkpoints").glob("checkpoint-*")):
    item = json.loads((path / "checkpoint.json").read_text())
    rows.append({"steps": item["actual_steps"], "phase": item["phase"],
        "MiB": sum(p.stat().st_size for p in path.iterdir() if p.is_file()) / 1024**2})
generations = pd.DataFrame(rows)
display(generations)
print(f"Checkpoint storage in this run: {generations.MiB.sum():.1f} MiB")

## 6. Storage and limits

Older generations are retained; nothing is deleted automatically. The default full budget has
32 completed rollout updates plus phase-start checkpoints. Sizes depend on workers, state and
history. Use measurements and leave room for packages, multiple learners and report artifacts.

A shutdown can lose unsaved work. A deleted volume can lose every local checkpoint.
No local-file implementation prevents that, and we are not adding S3.

CPU equality here is evidence for this environment. Run the lab on Studio too. Hardware,
GPU kernels and package changes can affect numerical reproducibility.

## 7. Recovery checklist

1. Reopen the same persistent project and Python environment.
2. Confirm the complete run folder and contract are present.
3. Resume the existing folder without changing its learning settings.
4. Read the recovered phase and transition count.
5. If checks fail, preserve the artifacts and investigate instead of silently starting over.
6. Treat the study run as finished only when the manifest says completed.

Read train/checkpoints.py and train/train_ppo.py to follow the implementation.